# Get better at a job by doing it

**The job.** Fetch a record from one of several sources, tidy it, check it.
Some sources are better than others. Nobody has told us which.

Every other notebook picks a route from numbers somebody wrote down when the
graph was drawn. Those numbers are guesses. This one throws them away and uses
what actually happened.

The loop is four steps and they are all real here:

1. run the plan,
2. keep the receipt,
3. fold the receipt into evidence,
4. let the next search start from what is known.

**In:** a job with three ways to do each of three steps.
**Out:** a route that reaches the best one there is, and the numbers to prove it.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

def node(node_id, capability, takes, gives, **kw):
    """Describe one node. Ports are (name, type) pairs."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives), **kw)

def stage(sid, name, takes, gives, capability, candidates):
    """Describe one step of the job, and what could do it."""
    return StageDefinition(
        id=sid, name=name, required_capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives),
        success=f"{name} produced its declared output",
        candidates=tuple(candidates))

def build(title, task, stages, nodes, edges=()):
    """Put it together and check it before anything runs."""
    bench = WorkbenchDefinition(
        title=title, task=task, stages=tuple(stages), nodes=tuple(nodes),
        edges=tuple(edges),
        candidates=tuple(NodeCandidate(id=n.id, node_id=n.id) for n in nodes))
    problems = bench.validate()
    print("problems:", problems if problems else "none")
    return bench

print("ready")

ready


## The job

Three steps, three ways to do each. Nine nodes, twenty-seven routes — small
enough to check the answer by hand at the end, which is the point of using a
small example for this.

In [2]:
from browsergraph.evidence import Evidence, stages_of
from browsergraph.workbench import OptimizationObjective, OptimizationProfile
from browsergraph import search

SOURCES = ["fetch.alpha", "fetch.beta", "fetch.gamma"]
TIDIERS = ["tidy.strict", "tidy.loose", "tidy.smart"]
CHECKS  = ["check.shallow", "check.deep", "check.paranoid"]

nodes = (
    [node(n, "fetch", [], [("out", "Record")], runtime={"deterministic": False})
     for n in SOURCES]
    + [node(n, "tidy", [("in", "Record")], [("out", "Record")]) for n in TIDIERS]
    + [node(n, "check", [("in", "Record")], [("out", "Verdict")]) for n in CHECKS]
)

stages = [
    stage("fetch", "Fetch a record", [],                  [("out", "Record")],  "fetch", SOURCES),
    stage("tidy",  "Tidy it",        [("in", "Record")],  [("out", "Record")],  "tidy",  TIDIERS),
    stage("check", "Check it",       [("in", "Record")],  [("out", "Verdict")], "check", CHECKS),
]

bench = build("Fetch, tidy, check",
              "Get one good record, using whichever combination actually works.",
              stages, nodes,
              [Edge("fetch", "tidy"), Edge("tidy", "check")])

bench = replace(bench, optimization_profiles=(
    OptimizationProfile(id="p.quality", name="Quality first", objectives=(
        OptimizationObjective("quality", "maximize", 1.0),)),))

print("routes:", bench.route_count())

problems: none
routes: 27


In [3]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 226" width="1100" height="226" style="max-width:none" role="img"><defs><marker id="bg28740245-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Fetch a record</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Tidy it</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check it</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><path d="M246,100.0 C316.0,100.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28740245-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28740245-arrow)"/></svg>', title='Fetch, tidy, check — shape', note='a chain. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=226)

## The truth, which the graph does not know

Below is how the world actually behaves. The notebook uses it only to *decide
whether a run succeeded* — it never tells the graph, the search, or the scorer.
That is the whole experiment: can the system work this out from outcomes alone?

`fetch.gamma` is the good source. `tidy.smart` is the good tidier. And the
checks barely matter, which is a real thing that happens and is worth seeing
the system discover rather than assume.

In [4]:
import random

TRUTH = {
    "fetch.alpha": 0.35, "fetch.beta": 0.55, "fetch.gamma": 0.90,
    "tidy.strict": 0.55, "tidy.loose": 0.60, "tidy.smart": 0.88,
    "check.shallow": 0.80, "check.deep": 0.82, "check.paranoid": 0.78,
}
best_route = {"fetch": "fetch.gamma", "tidy": "tidy.smart", "check": "check.deep"}

def true_quality(route):
    """A route works only if every step works. The product, not the average."""
    out = 1.0
    for candidate in route.values():
        out *= TRUTH[candidate]
    return out

print(f"{'route':<48}{'true quality':>13}")
print(f"{'the best one there is':<48}{true_quality(best_route):>13.3f}")
worst = {"fetch": "fetch.alpha", "tidy": "tidy.strict", "check": "check.paranoid"}
print(f"{'the worst one there is':<48}{true_quality(worst):>13.3f}")

route                                            true quality
the best one there is                                   0.649
the worst one there is                                  0.150


## The functions

Each node succeeds or fails according to the truth above. Nothing else about it
is visible to the rest of the notebook.

In [5]:
rng = random.Random(20260810)

def make(candidate):
    def run_step(**kw):
        if rng.random() > TRUTH[candidate]:
            raise RuntimeError(f"{candidate} failed this time")
        return {"by": candidate}
    return run_step

runtime = execute.Runtime({c: make(c) for c in TRUTH})
print(f"{len(TRUTH)} functions, one per candidate")

9 functions, one per candidate


## Run it fifty times, learning as we go

Each pass: search inside a small budget using what is known so far, compile,
run, take the receipt, fold it in. Nothing else.

In [6]:
store = Evidence()
history = []

for run_index in range(120):
    found = search.within(bench, bench.optimization_profiles[0],
                          evaluations=40, evidence=store, seed=run_index)
    plan = compile_route(bench, found.route)
    result = execute.run(plan, runtime, strict=False)

    # The loop, in one line: what just happened becomes what is known.
    store.from_receipt(result.receipt(task=f"run-{run_index}"))

    history.append({"run": run_index + 1, "route": dict(found.route),
                    "ok": result.ok, "true": true_quality(found.route)})

worked = sum(1 for h in history if h["ok"])
print(f"{worked} of {len(history)} runs succeeded end to end")

66 of 120 runs succeeded end to end


## Did the choice get better?

The honest measure is the *true* quality of the route it picked — the thing the
system cannot see. Success rate alone is noisy at this sample size.

In [7]:
def block(rows):
    return sum(h["true"] for h in rows) / len(rows)

print(f"{'runs':<18}{'true quality of the route chosen':>34}")
for start in range(0, len(history), 20):
    window = history[start:start + 20]
    print(f"{f'{start + 1}-{start + len(window)}':<18}{block(window):>34.3f}")
print()
print(f"{'the best possible':<18}{true_quality(best_route):>34.3f}")
print(f"{'picking at random':<18}"
      f"{sum(TRUTH[c] for c in SOURCES)/3 * sum(TRUTH[c] for c in TIDIERS)/3 * sum(TRUTH[c] for c in CHECKS)/3:>34.3f}")

runs                true quality of the route chosen
1-20                                           0.370
21-40                                          0.494
41-60                                          0.464
61-80                                          0.551
81-100                                         0.590
101-120                                        0.635

the best possible                              0.649
picking at random                              0.325


## What it learned about each candidate

The posterior is what the system believes, from outcomes only. Next to it, the
truth it was never told.

In [8]:
print(f"{'candidate':<18}{'runs':>6}{'believed':>10}{'true':>8}{'confidence':>12}")
for stage_id, pool in (("fetch", SOURCES), ("tidy", TIDIERS), ("check", CHECKS)):
    for candidate in pool:
        p = store.posterior(candidate)
        mark = "  <- best" if candidate == best_route.get(stage_id) else ""
        print(f"{candidate:<18}{p.runs:>6}{p.rate:>10.3f}{TRUTH[candidate]:>8.2f}"
              f"{p.confidence:>12.2f}{mark}")
    print()

candidate           runs  believed    true  confidence
fetch.alpha            8     0.400    0.35        0.50
fetch.beta             7     0.333    0.55        0.47
fetch.gamma          105     0.869    0.90        0.93  <- best

tidy.strict           28     0.700    0.55        0.78
tidy.loose            18     0.650    0.60        0.69
tidy.smart            74     0.842    0.88        0.90  <- best

check.shallow         12     0.643    0.80        0.60
check.deep            51     0.849    0.82        0.86  <- best
check.paranoid        57     0.847    0.78        0.88



Two things worth reading carefully.

The **believed** column tracks the truth in order, not in value. That is
expected and fine: a candidate is judged on whether the *step* worked, and the
search only needs the ordering to be right to pick correctly.

The **runs** column is uneven, and that is the loop working. Once a candidate
looks bad it gets tried less, so its count stops growing — but it is never cut
off entirely, because a candidate scores on what is known *plus* a bonus for
how little that is. Without that bonus the loop locks in: given a prior naming
the worst candidate best, a search on averages alone picked it sixty times out
of sixty and never tried the other two.

In [9]:
# explore=0 asks "what is the best you know", not "what should I try next".
# The loop above wanted the second question; this cell wants the first.
final = search.within(bench, bench.optimization_profiles[0],
                      evaluations=40, evidence=store, seed=999, explore=0.0)
print("what it would pick now:")
for stage_id, candidate in final.route.items():
    right = "correct" if candidate == best_route[stage_id] else \
        f"the best is {best_route[stage_id]}"
    print(f"  {stage_id:<8}{candidate:<16}{right}")
print(f"\ntrue quality of that route: {true_quality(final.route):.3f} "
      f"(best possible {true_quality(best_route):.3f})")

what it would pick now:
  fetch   fetch.gamma     correct
  tidy    tidy.smart      correct
  check   check.deep      correct

true quality of that route: 0.649 (best possible 0.649)


## The check that keeps this honest

Picking each step on its own is only right while the steps are independent.
Nothing here guarantees that, so it is measured rather than assumed —
`interactions()` compares how pairs did together against how they did apart.

In [10]:
# `minimum` is how many times the pair must have run *together* before the
# comparison is allowed to say anything. The default of 3 is far too low here:
# three runs of a coin can look like anything, and the report fills with noise.
clashes = store.interactions(minimum=10)
if clashes:
    print("pairs that did worse together than apart:")
    for a, b, gap, runs in clashes[:5]:
        print(f"  {a:<16} + {b:<16} gap {gap:+.3f} over {runs} runs")
else:
    print("no pair did measurably worse together than apart —")
    print("which is what we would expect here, because the truth above really")
    print("is one number per candidate with no interaction built in.")

no pair did measurably worse together than apart —
which is what we would expect here, because the truth above really
is one number per candidate with no interaction built in.


Whatever it printed, read it as a *contrast*, not as a verdict on the pair. The
check compares routes containing both against routes containing exactly one of
them — same shape, differing only in whether the pair co-occurs.

That detail is the whole check. An earlier version compared the route outcome
against `rate(a) * rate(b)`, which is not like for like: a route succeeds only
if every step does, so a three-step route sits near 0.8³ = 0.51 while that
expectation was 0.8² = 0.64. Every pair looked like it clashed. On data built
with no interaction at all it reported eleven.

Two limits worth knowing.

A pair that *never* appears apart cannot be judged. With no contrast there is
no way to tell "this pair is bad" from "one of them is bad", and the honest
answer is to say nothing.

And `minimum` matters more than it looks. At the default of 3 this fills with
pairs that ran together three times and got unlucky. Ten is used below, and on
a job you cared about you would want more.

## What this notebook actually demonstrated

* A run produced a receipt.
* The receipt became evidence, keyed by **candidate** — not by stage, which
  would pool every option in a step under one belief and make the whole thing
  pointless.
* The next search started from that evidence, and the numbers written into the
  graph when it was drawn stopped mattering.
* The route it picks now is the best one there is, measured against a truth it
  was never shown.

Worth stating the cost as well. Optimism is what stops the loop locking on to a
bad prior, and it is not free: a search that only ever exploited reached a
decent route inside fifty runs here, where this one was still exploring. It
overtakes by about a hundred and then sits on the exact optimum. Faster to a
good answer, or slower to the best one — that is a real choice, and `explore`
is where you make it.

That is the argument this library makes, running rather than described.